# vForge — 01. Dataset generation

Generate a synthetic instruction-tuning dataset with an open-weight LLM.

**Sprint topic:** Google TPU Sprint Q1 2026 — vLLM benchmarking TPU vs GPU.

Open-weight providers (Ollama, Together AI) are recommended — outputs are safe to use for fine-tuning.

## 1. Install

In [ ]:
!pip -q install httpx tenacity

## 2. Configure provider

Set `TOGETHER_API_KEY` (or use Ollama locally). Together's open-model endpoint works in Colab without local resources.

In [ ]:
import os
os.environ['TOGETHER_API_KEY'] = 'YOUR_KEY_HERE'  # set me
PROVIDER = 'together'
MODEL = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'

## 3. Generate

In [ ]:
import httpx, json, asyncio

SYSTEM = (
  'You generate diverse instruction/response pairs for fine-tuning.\n'
  'Output ONLY raw JSONL with keys instruction, input, output.'
)

async def gen(domain: str, description: str, n: int):
    rows = []
    while len(rows) < n:
        batch_n = min(20, n - len(rows))
        prompt = f'Domain: {domain}\nGoal: {description}\nGenerate exactly {batch_n} JSONL rows.'
        async with httpx.AsyncClient(timeout=600) as c:
            r = await c.post(
              'https://api.together.xyz/v1/chat/completions',
              headers={'Authorization': f"Bearer {os.environ['TOGETHER_API_KEY']}"},
              json={'model': MODEL,
                    'messages': [{'role':'system','content':SYSTEM},
                                 {'role':'user','content':prompt}],
                    'max_tokens': 4096, 'temperature': 0.8})
            r.raise_for_status()
            text = r.json()['choices'][0]['message']['content']
        for line in text.splitlines():
            line = line.strip().lstrip('```jsonl').lstrip('```json').lstrip('```').rstrip('```')
            try:
                obj = json.loads(line)
                if 'instruction' in obj and 'output' in obj:
                    rows.append({'instruction': obj['instruction'],
                                 'input': obj.get('input',''),
                                 'output': obj['output']})
            except Exception:
                continue
    return rows[:n]

rows = await gen('code', 'Pandas one-liner code completions for data analysis', n=200)
print(len(rows), 'rows')
rows[:2]

## 4. Save and inspect

In [ ]:
from pathlib import Path
Path('data').mkdir(exist_ok=True)
with open('data/code.jsonl','w') as f:
    for r in rows: f.write(json.dumps(r) + '\n')
!head -2 data/code.jsonl

## 5. Push to HuggingFace (optional)

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi(token='hf_...')
# api.upload_file(path_or_fileobj='data/code.jsonl', path_in_repo='train.jsonl',
#                 repo_id='you/your-dataset', repo_type='dataset')